In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Re‑define the run_plsda function exactly as in your line‑broadening script
# (I assume it's already defined in your current script; if not, copy it here)

# ── Load 400 MHz binned data ──
df_400 = pd.read_csv("data/NMR Binning Data commercial honey 400MHz.xlsx - Sheet1.csv")
y_400 = df_400["Type"]
X_400 = df_400.drop(columns=["Samplecode", "Species", "Type"])

# Water removal (same as original)
bin_ppm = X_400.columns.astype(float)
keep = (bin_ppm < 4.68) | (bin_ppm > 5.03)
X_400 = X_400.loc[:, keep]

# Label encode
le = LabelEncoder()
y_400_enc = le.fit_transform(y_400)

# Pareto scaling
X_np = X_400.values
scaler = StandardScaler(with_mean=True, with_std=False)
X_centered = scaler.fit_transform(X_np)
std = X_np.std(axis=0)
std[std == 0] = 1
X_pareto_400 = X_centered / np.sqrt(std)

# Run PLS‑DA
metrics_400 = run_plsda(X_pareto_400, y_400_enc)
print("400 MHz binned data:")
print(f"  CV accuracy: {metrics_400['cv_mean']*100:.1f}% ± {metrics_400['cv_std']*100:.1f}%")
print(f"  OOB accuracy: {metrics_400['oob_mean']*100:.1f}% ± {metrics_400['oob_std']*100:.1f}%")

In [ ]:
import nmrglue as ng

# Test load a single 400 MHz FID
path = "data/400MHz/H1/10"   # folder containing 'fid' file
try:
    dic, fid = ng.bruker.read(path)
    print("Successfully read FID for H1")
    print("  Number of points:", len(fid))
    print("  First 5 complex points (real):", fid[:5].real)
    # Print some acquisition parameters
    sw = dic["acqus"].get("SW_h", dic["acqus"].get("SW", "?"))
    td = dic["acqus"].get("TD", "?")
    print("  Spectral width (Hz):", sw)
    print("  Time domain size:", td)
except Exception as e:
    print("Error reading FID:", e)

In [ ]:
import pandas as pd

df400 = pd.read_csv("data/NMR Binning Data commercial honey 400MHz.xlsx - Sheet1.csv")
df700 = pd.read_csv("data/NMR Binning Data commercial honey 700MHz.xlsx - Sheet1.csv")

print("400 MHz shape:", df400.shape)
print("700 MHz shape:", df700.shape)
print("400 MHz columns (first 5):", df400.columns[:5].tolist())
print("700 MHz columns (first 5):", df700.columns[:5].tolist())
print("Sample codes 400:", df400.iloc[:,0].tolist())
print("Sample codes 700:", df700.iloc[:,0].tolist())